# Monitor de Riesgo Bancario Chileno
## Notebook 3: Dashboard Interactivo de Riesgo

> Dashboard interactivo con Plotly que consolida todos los analisis del proyecto en una vista ejecutiva,
> equivalente al tipo de reporte que usa una mesa de riesgo o la propia CMF para monitoreo.

---
## 0. Configuracion

In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROC_DIR   = Path('..') / 'data' / 'processed'
REPORT_DIR = Path('..') / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(PROC_DIR / 'dataset_maestro.parquet')
df['fecha'] = pd.to_datetime(df['fecha'])

mora_consumo_sis  = df[df['tiene_cartera_consumo']].groupby('fecha')['mora_consumo'].mean()
mora_vivienda_sis = df[df['tiene_cartera_vivienda']].groupby('fecha')['mora_vivienda'].mean()

df_sistema = df.groupby('fecha').agg(
    mora_total     = ('mora_total',     'mean'),
    mora_comercial = ('mora_comercial', 'mean'),
    tpm            = ('tpm',            'first'),
    imacec         = ('imacec',         'first'),
    ciclo          = ('ciclo',          'first'),
).reset_index()

df_sistema = df_sistema.merge(mora_consumo_sis.rename('mora_consumo'),   on='fecha', how='left')
df_sistema = df_sistema.merge(mora_vivienda_sis.rename('mora_vivienda'), on='fecha', how='left')

COLORES = {
    'fondo'   : '#0d1117',
    'panel'   : '#161b22',
    'borde'   : '#30363d',
    'texto'   : '#c9d1d9',
    'azul'    : '#58a6ff',
    'rojo'    : '#f85149',
    'verde'   : '#3fb950',
    'amarillo': '#d29922',
    'naranja' : '#db6d28',
}

print('Datos cargados para el dashboard')
print(f'Bancos: {df["banco"].nunique()} | Meses: {df["fecha"].nunique()}')
print(f'Bancos retail con cartera consumo  : {df[df["tiene_cartera_consumo"]]["banco"].nunique()}')
print(f'Bancos retail con cartera vivienda : {df[df["tiene_cartera_vivienda"]]["banco"].nunique()}')


Datos cargados para el dashboard
Bancos: 25 | Meses: 122
Bancos retail con cartera consumo  : 16
Bancos retail con cartera vivienda : 14


---
## 1. Panel Principal — Vista del Sistema

In [5]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Morosidad 90+ dias - Sistema Bancario',
        'Tasa de Politica Monetaria (TPM)',
        'Morosidad por Tipo de Cartera',
        'IMACEC - Actividad Economica',
        'Distribucion de Morosidad por Banco',
        'Correlacion TPM vs Morosidad',
    ],
    specs=[
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'bar'},     {'type': 'scatter'}],
    ],
    vertical_spacing=0.10,
    horizontal_spacing=0.08,
)

# 1. Morosidad total
fig.add_trace(go.Scatter(
    x=df_sistema['fecha'],
    y=df_sistema['mora_total'],
    name='Mora Total',
    line=dict(color=COLORES['rojo'], width=2.5),
    fill='tozeroy',
    fillcolor='rgba(248,81,73,0.12)',
    hovertemplate='%{x|%b %Y}<br>Morosidad: %{y:.2f}%<extra></extra>',
), row=1, col=1)

# 2. TPM
fig.add_trace(go.Scatter(
    x=df_sistema['fecha'],
    y=df_sistema['tpm'],
    name='TPM',
    line=dict(color=COLORES['azul'], width=2.5, shape='hv'),
    fill='tozeroy',
    fillcolor='rgba(88,166,255,0.10)',
    hovertemplate='%{x|%b %Y}<br>TPM: %{y:.2f}%<extra></extra>',
), row=1, col=2)

# 3. Por cartera
carteras = [
    ('mora_consumo',   'Consumo',   COLORES['rojo']),
    ('mora_comercial', 'Comercial', COLORES['amarillo']),
    ('mora_vivienda',  'Vivienda',  COLORES['verde']),
]
for col_var, nombre, color in carteras:
    fig.add_trace(go.Scatter(
        x=df_sistema['fecha'],
        y=df_sistema[col_var],
        name=nombre,
        line=dict(color=color, width=1.8),
        hovertemplate=f'{nombre}: %{{y:.2f}}%<extra></extra>',
    ), row=2, col=1)

# 4. IMACEC YoY
imacec_yoy = df_sistema['imacec'].pct_change(12) * 100
fig.add_trace(go.Scatter(
    x=df_sistema['fecha'],
    y=imacec_yoy,
    name='IMACEC YoY%',
    line=dict(color=COLORES['verde'], width=2),
    fill='tozeroy',
    fillcolor='rgba(63,185,80,0.10)',
    hovertemplate='%{x|%b %Y}<br>IMACEC: %{y:.1f}%<extra></extra>',
), row=2, col=2)
fig.add_hline(y=0, line_color=COLORES['borde'], line_dash='dash', row=2, col=2)

# 5. Ranking por banco
ultimo_mes    = df['fecha'].max()
bancos_retail = df[df['tipo_banco'] == 'Retail']['banco'].unique()
df_ult = (
    df[(df['fecha'] == ultimo_mes) & (df['banco'].isin(bancos_retail))]
    .groupby('banco')['mora_total'].mean()
    .sort_values(ascending=True)
)
colores_bars = [
    COLORES['rojo'] if v > 3 else COLORES['amarillo'] if v > 2 else COLORES['verde']
    for v in df_ult.values
]
fig.add_trace(go.Bar(
    x=df_ult.values,
    y=df_ult.index,
    orientation='h',
    name='Mora por Banco',
    marker_color=colores_bars,
    hovertemplate='%{y}: %{x:.2f}%<extra></extra>',
), row=3, col=1)

# 6. Scatter TPM vs Morosidad
df_sc = df_sistema.dropna(subset=['tpm', 'mora_total'])
ciclos_colores_p = {
    'Pre-Estallido (2015-2019)'     : COLORES['azul'],
    'Estallido Social'              : COLORES['amarillo'],
    'COVID-19'                      : COLORES['rojo'],
    'Ciclo Inflacionario / Alza TPM': COLORES['naranja'],
    'Normalizacion Monetaria'       : COLORES['verde'],
}
for ciclo, color in ciclos_colores_p.items():
    mask = df_sc['ciclo'] == ciclo
    if mask.any():
        fig.add_trace(go.Scatter(
            x=df_sc.loc[mask, 'tpm'],
            y=df_sc.loc[mask, 'mora_total'],
            mode='markers',
            name=ciclo,
            marker=dict(color=color, size=6, opacity=0.75),
            hovertemplate=f'TPM: %{{x:.1f}}% | Mora: %{{y:.2f}}%<extra></extra>',
        ), row=3, col=2)

# Layout
fig.update_layout(
    title=dict(
        text='<b>Monitor de Riesgo Bancario Chileno</b><br><sup>Fuente: CMF Chile & Banco Central de Chile | 2015-2025</sup>',
        font=dict(size=18, color=COLORES['texto']),
        x=0.5
    ),
    height=1000,
    paper_bgcolor=COLORES['fondo'],
    plot_bgcolor =COLORES['panel'],
    font=dict(family='monospace', color=COLORES['texto'], size=11),
    legend=dict(
        bgcolor=COLORES['panel'],
        bordercolor=COLORES['borde'],
        borderwidth=1,
        font=dict(size=9),
    ),
    hovermode='x unified',
)

for i in range(1, 4):
    for j in range(1, 3):
        fig.update_xaxes(gridcolor=COLORES['borde'], showgrid=True, row=i, col=j)
        fig.update_yaxes(gridcolor=COLORES['borde'], showgrid=True, row=i, col=j)

fig.update_yaxes(ticksuffix='%', row=1, col=1)
fig.update_yaxes(ticksuffix='%', row=1, col=2)
fig.update_yaxes(ticksuffix='%', row=2, col=1)
fig.update_yaxes(ticksuffix='%', row=2, col=2)
fig.update_xaxes(ticksuffix='%', row=3, col=1)

fig.write_html(str(REPORT_DIR / 'dashboard_interactivo.html'))
fig.show()
print('Dashboard guardado en reports/dashboard_interactivo.html')


Dashboard guardado en reports/dashboard_interactivo.html


---
## 2. Semaforo de Riesgo por Institucion

In [6]:
def clasificar_riesgo(mora):
    if mora >= 4.0:   return 'ALTO',  '#f85149', 'ROJO'
    elif mora >= 2.5: return 'MEDIO', '#d29922', 'AMARILLO'
    else:             return 'BAJO',  '#3fb950', 'VERDE'

df_semaforo = (
    df[(df['fecha'] == ultimo_mes) & (df['banco'].isin(bancos_retail))]
    .groupby('banco')[['mora_total', 'mora_consumo', 'mora_comercial', 'mora_vivienda']]
    .mean()
    .reset_index()
    .sort_values('mora_total', ascending=False)
)

def fmt(x):
    return f'{x:.2f}%' if pd.notna(x) else 'N/A'

niveles = [clasificar_riesgo(m) for m in df_semaforo['mora_total']]
df_semaforo['nivel'] = [n[0] for n in niveles]
df_semaforo['color'] = [n[1] for n in niveles]
df_semaforo['icono'] = [n[2] for n in niveles]

fig_sem = go.Figure(data=[
    go.Table(
        header=dict(
            values=['<b>Institucion</b>', '<b>Mora Total</b>', '<b>Consumo</b>',
                    '<b>Comercial</b>', '<b>Vivienda</b>', '<b>Nivel de Riesgo</b>'],
            fill_color=COLORES['borde'],
            font=dict(color=COLORES['texto'], size=12, family='monospace'),
            align='center',
            height=36,
        ),
        cells=dict(
            values=[
                df_semaforo['banco'],
                df_semaforo['mora_total'].apply(fmt),
                df_semaforo['mora_consumo'].apply(fmt),
                df_semaforo['mora_comercial'].apply(fmt),
                df_semaforo['mora_vivienda'].apply(fmt),
                [f'{i} - {n}' for i, n in zip(df_semaforo['icono'], df_semaforo['nivel'])],
            ],
            fill_color=[
                [COLORES['panel']] * len(df_semaforo),
                [COLORES['panel']] * len(df_semaforo),
                [COLORES['panel']] * len(df_semaforo),
                [COLORES['panel']] * len(df_semaforo),
                [COLORES['panel']] * len(df_semaforo),
                df_semaforo['color'].tolist(),
            ],
            font=dict(color=COLORES['texto'], size=11, family='monospace'),
            align='center',
            height=32,
        )
    )
])

fig_sem.update_layout(
    title=dict(
        text=f'<b>Semaforo de Riesgo - Sistema Bancario Chileno</b><br><sup>Periodo: {ultimo_mes.strftime("%B %Y")} | Fuente: CMF Chile | Solo bancos retail</sup>',
        font=dict(size=16, color=COLORES['texto']),
        x=0.5
    ),
    paper_bgcolor=COLORES['fondo'],
    font=dict(family='monospace', color=COLORES['texto']),
    height=460,
)

fig_sem.write_html(str(REPORT_DIR / 'semaforo_riesgo.html'))
fig_sem.show()
print('Semaforo guardado en reports/semaforo_riesgo.html')


Semaforo guardado en reports/semaforo_riesgo.html
